In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams["font.family"] = 'NanumGothic'
PATH = '../P1_Data/03.서비스권역내(국토정보플랫폼)/'
PATH2 = '../P1_Data/04.서비스권역내(기존데이터)/'
path_1 = '../P1_Data/01.External/양주시 분석 기초데이터/'
path_2 = '../P1_Data/01.External/양주시 분석 기초데이터(2차)/'

In [ ]:
# 행정경계 데이터
boundary = gpd.read_file('../P1_Data/01.External/지도데이터/Z_SOP_BND_ADM_DONG_PG/Z_SOP_BND_ADM_DONG_PG.shp', encoding = 'cp949')
boundary = boundary.to_crs(epsg=5179)
yang_boundary = boundary[boundary['ADM_DR_CD'].astype(str).str.startswith('3126')] #양주시만 추출

#행정동코드
HJD_cd = pd.read_excel('../P1_Data/01.External/KIKcd_H.20230501.xlsx')
HJD_cd['행정동코드'] = HJD_cd['행정동코드'].astype('str')
HJD_cd['행정동코드'] = HJD_cd['행정동코드'].apply(lambda x : x[0:8])
HJD_cd.rename(columns={'읍면동명': '행정기관'}, inplace=True)
HJD_cd_yang = HJD_cd[HJD_cd['시군구명'] == '양주시']

pre_boundary = pd.merge(yang_boundary,HJD_cd_yang,left_on='ADM_DR_NM',right_on='행정기관')
pre_boundary = pre_boundary[['행정동코드','geometry']]
pre_boundary.columns = ['hjd_cd','geometry']


#총인구
pop = pd.read_excel('../P1_Data/01.External/양주시 분석 기초데이터/1. 인구구조/양주시 인구 및 세대현황(2023331기준)(행정동별, 법정동별).xlsx',skiprows=5)
pop = pop[['Unnamed: 1','계', '남', '여', '계.1', '남.1','여.1','Unnamed: 9','Unnamed: 10', 'Unnamed: 11']]
pop.columns = ['행정기관','총인구','남자인구수','여자인구수','구성비율합계','남자비율','여자비율','성비','세대수','세대당인구']

pop_code = pd.merge(HJD_cd_yang,pop, on='행정기관')
pop_code.drop_duplicates('행정기관')
#pop.to_csv(pre_path+'pre_pop.csv',index=False,encoding='cp949')


hjd_geo0 = pd.merge(pre_boundary,pop_code,left_on='hjd_cd',right_on='행정동코드')


hjd_geo = hjd_geo0[['hjd_cd','총인구','geometry']]
hjd_geo.columns = ['hjd_cd','total_pop','geometry']
hjd_geo['total_pop'] = hjd_geo['total_pop'].astype(float)

#바탕용 행정경계 생성
hjd_geo2 = hjd_geo0[['hjd_cd','행정기관','총인구','geometry']]
hjd_geo2.columns = ['hjd_cd','hjd_nm','total_pop','geometry']
#hjd_geo2.to_file('../P1_Data/02.Pretreatment/hjd_geo2.gpkg', driver='GPKG')

In [ ]:
#노인인구 & 영유아인구 매핑
people5 = pd.read_excel(path_1+'1. 인구구조/통계청_행정구역_읍면동_별_5세별_주민등록인구_2013__202303.xlsx') # 5세단위 인구수
people5 = people5[people5['행정구역(동읍면)별'].notnull()]
selected_column = people5['행정구역(동읍면)별']
concat_df = pd.concat([selected_column, people5.loc[:, '2023.03':]], axis=1)
people_2023 = concat_df.reset_index(drop=True)
people_2023.columns = people_2023.iloc[0]
people_2023 = people_2023[1:]
people_2023.rename(columns={'행정구역(동읍면)별': '행정기관','계':'총인구'}, inplace=True)
people_2023.loc[:, '0 - 4세':'100+'] = people_2023.loc[:, '0 - 4세':'100+'].apply(lambda x: x.astype(int))

#고령화 인구
age_col_65 = people_2023.loc[:, '65 - 69세':'100+']
#영유아 인구
age_col_4 = people_2023.loc[:,'0 - 4세']

# 선택한 컬럼들 가로 방향으로 더하기
filter_age = pd.DataFrame({
    'hjd_nm': people_2023['행정기관'],
    'old_pop': age_col_65.sum(axis=1),
    'young_pop' : people_2023['0 - 4세']
})

filter_age['hjd_nm'] = filter_age['hjd_nm'].str.strip()
filter_age['old_pop'] = filter_age['old_pop'].astype(int)
filter_age['young_pop'] = filter_age['young_pop'].astype(int)

In [ ]:
hjd_geo3 = pd.merge(hjd_geo2,filter_age,on='hjd_nm')
hjd_geo3= hjd_geo3[['hjd_cd','hjd_nm','total_pop','old_pop','young_pop','geometry']]

In [ ]:
#읍면동별 면적 매핑
area = pd.read_excel(path_1+'2. 경제활력 및 기업현황/자체통계_양주통계연보_2020년기준/02토지및기후.xlsx',sheet_name='2. 행정구역')
area = area[['2.  행   정   구   역','Unnamed: 1']]
area.columns = ['hjd_nm','area']
hjd_geo4 = pd.merge(hjd_geo3,area,on='hjd_nm')
hjd_geo4['area'] = hjd_geo4['area'].astype(float)
hjd_geo4= hjd_geo4[['hjd_cd','hjd_nm','total_pop','old_pop','young_pop','area','geometry']]

In [ ]:
#최종 바탕 경계면 완성 ( 권역별코드 매핑 )
region_cd = pd.read_excel('../P1_Data/02.Pretreatment/권역별_코드.xlsx', dtype={'권역코드': int})
region_cd.columns = ['rei_cd','rei_nm','hjd_nm']
result_boudary = pd.merge(region_cd,hjd_geo4,on='hjd_nm')
result_boudary = result_boudary[['rei_cd','rei_nm','hjd_cd','hjd_nm','total_pop','old_pop','young_pop','area','geometry']]

In [ ]:
result_boudary = gpd.GeoDataFrame(result_boudary, geometry='geometry')
#result_boudary.to_file('../P3_Result/03.SHP/서비스권역내인구비율/result_boundary.gpkg', driver='GPKG')

In [ ]:
result_rei_boudary = result_boudary.groupby(['rei_cd', 'rei_nm']).agg({
    'total_pop': 'sum',
    'old_pop': 'sum',
    'young_pop': 'sum',
    'area': 'sum',
    'geometry': lambda x: x.unary_union
}).reset_index()

In [ ]:
result_rei_boudary = gpd.GeoDataFrame(result_rei_boudary, geometry='geometry')
#result_rei_boudary.to_file('../P3_Result/03.SHP/서비스권역내인구비율/result_rei_boudary.gpkg', driver='GPKG')

In [ ]:
# 총인구 기준 100m 격자
#행정동별 코드가 포함된 양주시 100m 격자 (soc작업시, 생성_)
layer = gpd.read_file('../P1_Data/02.Pretreatment/grid_100.shp', encoding='cp949')

In [ ]:
# 고령인구 기준 100m 격자
old_layer = gpd.read_file('../P1_Data/01.External/지도데이터/(B100)국토통계_인구정보-고령 인구 수(전체)-(격자) 100M_경기도 양주시_202304/nlsp_021001010.shp',encoding='cp949')
# 격자 법정동 이름 병합
over = gpd.overlay(old_layer,hjd_geo2,how='intersection')
over['area'] = over['geometry'].area
over.sort_values('area',ascending = False,inplace=True)
over.drop_duplicates(subset='gid',inplace=True,keep='first')
grid_hjd = over[['gid','hjd_cd']]
old_grid_100 = pd.merge(old_layer[['gid','geometry','val']],grid_hjd,on='gid')
old_grid_100 = gpd.GeoDataFrame(old_grid_100,geometry='geometry')

In [ ]:
# 유아인구 기준 100m 격자 (국토정보플랫폼 국토정보맵 유아인구수(1~7세))
young_layer = gpd.read_file('../P1_Data/01.External/지도데이터/(B100)국토통계_인구정보-유아 인구 수(전체)-(격자) 100M_경기도 양주시_202304/nlsp_021001013.shp',encoding='cp949')
# 격자 법정동 이름 병합
over = gpd.overlay(young_layer,hjd_geo2,how='intersection')
over['area'] = over['geometry'].area
over.sort_values('area',ascending = False,inplace=True)
over.drop_duplicates(subset='gid',inplace=True,keep='first')
grid_hjd = over[['gid','hjd_cd']]
young_grid_100 = pd.merge(young_layer[['gid','geometry','val']],grid_hjd,on='gid')
young_grid_100 = gpd.GeoDataFrame(young_grid_100,geometry='geometry')

In [ ]:
# 학령인구 기준 100m 격자 ( 국토정보플랫폼 학생 [초등학생,중학생,고등학생] 데이터 합침)
student1 = gpd.read_file('../P1_Data/01.External/지도데이터/(B100)국토통계_인구정보-초등학생 인구 수(전체)-(격자) 100M_경기도 양주시_202304/nlsp_021001016.shp',encoding='cp949') #초등학생
student2 = gpd.read_file('../P1_Data/01.External/지도데이터/(B100)국토통계_인구정보-중학생 인구 수(전체)-(격자) 100M_경기도 양주시_202304/nlsp_021001019.shp',encoding='cp949') #중학생
student3 = gpd.read_file('../P1_Data/01.External/지도데이터/(B100)국토통계_인구정보-고등학생 인구 수(전체)-(격자) 100M_경기도 양주시_202304/nlsp_021001022.shp',encoding='cp949') #고등학생
merged = gpd.GeoDataFrame(pd.concat([student1, student2, student3], ignore_index=True))

In [ ]:
studend_layer = merged.groupby(['gid','geometry'])['val'].sum().reset_index()
studend_layer = gpd.GeoDataFrame(studend_layer,geometry='geometry')

In [ ]:
# 격자 법정동 이름 병합
over = gpd.overlay(studend_layer,hjd_geo2,how='intersection')
over['area'] = over['geometry'].area
over.sort_values('area',ascending = False,inplace=True)
over.drop_duplicates(subset='gid',inplace=True,keep='first')
grid_hjd = over[['gid','hjd_cd']]
studend_grid_100 = pd.merge(studend_layer[['gid','geometry','val']],grid_hjd,on='gid')
studend_grid_100 = gpd.GeoDataFrame(studend_grid_100,geometry='geometry')

In [ ]:
# 격자 저장
layer.to_file('../P1_Data/02.Pretreatment/격자/total_pop_grid100.gpkg', driver='GPKG') #총인구 100m격자
old_grid_100.to_file('../P1_Data/02.Pretreatment/격자/old_grid100.gpkg', driver='GPKG') #고령인구 100m 격자
young_grid_100.to_file('../P1_Data/02.Pretreatment/격자/young_grid100.gpkg', driver='GPKG') #유아인구 100m 격자
studend_grid_100.to_file('../P1_Data/02.Pretreatment/격자/studend_grid100.gpkg', driver='GPKG') #학령인구 100m 격자

In [ ]:
#권역별 격자 생성
rei_cd = pd.read_excel('../P1_Data/02.Pretreatment/권역별_코드_격자생성.xlsx')
# 특정 컬럼의 데이터 타입을 str로 변경
rei_cd['hjd_cd'] = rei_cd['hjd_cd'].astype(str)
rei_cd

In [ ]:
layer_rei = pd.merge(rei_cd,layer,on='hjd_cd')
layer_rei = layer_rei[['gid','val','권역코드','권역명','geometry']]
layer_rei = gpd.GeoDataFrame(layer_rei,geometry='geometry')

In [ ]:
old_grid_100 = old_grid_100[['gid','val','hjd_cd','geometry']]
old_grid_rei = pd.merge(rei_cd,old_grid_100,on='hjd_cd')
old_grid_rei = old_grid_rei[['gid','val','권역코드','권역명','geometry']]
old_grid_rei = gpd.GeoDataFrame(old_grid_rei,geometry='geometry')

In [ ]:
young_grid_100 = young_grid_100[['gid','val','hjd_cd','geometry']]
young_grid_rei = pd.merge(rei_cd,young_grid_100,on='hjd_cd')
young_grid_rei = young_grid_rei[['gid','val','권역코드','권역명','geometry']]
young_grid_rei = gpd.GeoDataFrame(young_grid_rei,geometry='geometry')

In [ ]:
studend_grid_100 = studend_grid_100[['gid','val','hjd_cd','geometry']]
studend_grid_rei = pd.merge(rei_cd,studend_grid_100,on='hjd_cd')
studend_grid_rei = studend_grid_rei[['gid','val','권역코드','권역명','geometry']]
studend_grid_rei = gpd.GeoDataFrame(studend_grid_rei,geometry='geometry')

In [ ]:
#권역별 격자 저장
layer_rei.to_file('../P1_Data/02.Pretreatment/격자/rei_total_pop_grid100.gpkg', driver='GPKG') #총인구 100m격자
old_grid_rei.to_file('../P1_Data/02.Pretreatment/격자/rei_old_grid100.gpkg', driver='GPKG') #고령인구 100m 격자
young_grid_rei.to_file('../P1_Data/02.Pretreatment/격자/rei_young_grid100.gpkg', driver='GPKG') #유아인구 100m 격자
studend_grid_rei.to_file('../P1_Data/02.Pretreatment/격자/rei_studend_grid100.gpkg', driver='GPKG') #학령인구 100m 격자